<a href="https://colab.research.google.com/github/gflamiano/SJU-CSC-672-ML/blob/main/HW5/HW5_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 Analysis

### 1. Explain how implicit missing values impact medical datasets, and justify your choice of imputation strategy when handling these zeros compared to outright row deletion.

Implicit missing values are placeholders that look like real data. In this dataset, 0 appears in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`, where a true zero is impossible. Pandas reported 768 non-null values in every column, so nothing warned us. The zeros also changed the statistics. In the raw `describe()` output, `Insulin` had a mean of 79.8 and a median of only 30.5 because so many zeros were included, and `SkinThickness` had a 25th percentile of 0. If left in, these zeros would distort scaling and teach the models that a blood pressure of 0 is a real measurement. In medical data this is risky because values are rarely missing at random. A test may be skipped for some patients because of their condition, so the placeholder hides information about how the data was collected.

I replaced the zeros with `NaN` and filled them with the median instead of deleting rows. The zero counts show why deletion is a poor choice: `Insulin` had 374 zeros (48.7%) and `SkinThickness` had 227 (29.6%). Deleting just the rows with a zero `Insulin` would already remove at least 374 of the 768 patients, nearly half the data, and removing every row with any zero would remove even more. That would shrink the training set a lot and could bias it, since the patients who remain may not represent the whole group. Deletion also does nothing for a new patient whose measurement is missing at prediction time. I chose the median over the mean because these columns are skewed, and a few large values (the maximum `Insulin` is 846) would pull the mean up while the median stays stable. The medians used were 117 for `Glucose`, 72 for `BloodPressure`, 29 for `SkinThickness`, 125 for `Insulin`, and 32.3 for `BMI`. The downside of imputation is that every filled value is identical, which understates the real variation in the data, especially for `Insulin`, where nearly half the values were filled.

### 2. Contrast how hard voting aggregates predictions versus soft voting. Why do well-calibrated probability estimates matter when implementing soft voting across heterogeneous models?

Hard voting collects the class each model predicts, and the class chosen by most models wins. It only looks at the final decisions, not at how confident each model was. Soft voting averages the predicted probabilities from all models and picks the class with the highest average, so confidence matters. One very confident model can outweigh two unsure ones.

In our results, the two methods performed almost the same. Both reached a test accuracy of 0.708. Hard voting had a diabetic recall of 0.46 and soft voting had 0.48, which is a difference of about one patient out of 54. So soft voting did not give a real advantage here. One reason is that with only three models that often agree, the averaged probabilities rarely change the majority result.

Calibration matters because soft voting treats every model's probabilities as if they mean the same thing. Different models produce probabilities of very different quality. Logistic Regression is usually well calibrated, the SVC's probabilities are only an approximation added on top of the model, and a fully grown decision tree gives mostly 0 or 1, which are extreme values. When one model is overconfident, its extreme probabilities can dominate the average even if it is not the most reliable model, and the decision tree in our ensemble can end up acting like a hard vote. Calibrating the models, limiting tree depth, or giving the models different weights can help.

### 3. What is the role of out-of-bag evaluation in Bagging classifiers, and why does an OOB score serve as an efficient alternative to traditional cross-validation splits during ensemble training?

In Bagging, each tree is trained on a random sample of the training data drawn with replacement. Each sample contains only about 63% of the unique rows, so about 37% are left out of that tree and are called out-of-bag. To get the OOB score, each training row is predicted using only the trees that never saw it, and those predictions are compared with the true labels. This gives an honest accuracy estimate without using any separate data.

It is a cheaper alternative to cross-validation because it comes from a single training run. K-fold cross-validation has to retrain the whole ensemble k times, and it needs data to be held back for each fold. With OOB evaluation, every row is used for training and every row still gets a fair prediction. In our results, the OOB score was 0.748 and the test accuracy was 0.727. The difference is about 2 points, which is roughly 3 patients out of the 154 in the test set. The two estimates agree well, which suggests the OOB score is a reliable guide. It is also based on all 614 training rows, so it is less noisy than a test score from only 154 patients. It does not replace a final test set, but it is a quick and free check during training.

### 4. How does introducing random feature subsetting in Random Forests differ from standard Bagging, and how does this feature diversity impact model variance and generalization?

Standard Bagging trains each tree on a random sample of rows, but every split can still look at all the features. A Random Forest adds a second layer of randomness: at each split, only a random subset of the features is considered (with `max_features="sqrt"`, that is 2 of the 8). In our data, `Glucose` is by far the strongest feature, with an importance of about 0.27, followed by `BMI` at about 0.16. In plain Bagging, these two features would likely win the top splits in most trees, which makes the trees similar and makes them repeat the same mistakes. Feature subsetting forces the trees to use other features too, so they become more different from each other. The importance plot shows this spread: even the weakest feature, `SkinThickness`, still has an importance of about 0.07.

This matters because averaging only reduces variance well when the trees make different errors. Adding more trees to a Bagging model cannot fix the part of the variance that comes from the trees being similar, but making the trees less alike does. Each individual tree gets slightly weaker, but the drop in the ensemble's variance usually outweighs that loss. In our results, the Random Forest reached a test accuracy of 0.779 compared with 0.727 for Bagging, an improvement of about 5 points, or roughly 8 patients. With a test set of only 154 patients, a difference of this size could partly be noise, but it points in the expected direction.

### 5. Discuss the mechanics of sequential sample weighting in AdaBoost. Based on your training versus test accuracy results, did AdaBoost exhibit signs of overfitting, and how can hyperparameter adjustments mitigate this?

AdaBoost builds its weak learners (here, decision stumps) one after another. At the start, every training sample has the same weight. After each stump is trained, the weights of the samples it got wrong are increased and the weights of the ones it got right are decreased, so the next stump focuses on the hardest cases. Each stump also gets a vote weight based on its accuracy, with more accurate stumps counting for more. The final prediction is a weighted vote of all the stumps.

Our training accuracy was 0.822 and our test accuracy was 0.760, a gap of about 6 points. The plot shows how this gap formed. Both lines rose quickly during the first 40 stumps, with some bouncing up and down around stumps 20 to 35. After that, the training accuracy stayed near 0.82 and the test accuracy stayed near 0.76, and the gap did not grow even as the number of stumps went to 200. This means AdaBoost showed only mild overfitting, and adding more stumps did not make it worse. The gap looks like an ordinary difference between training and unseen data, which is expected for simple stump models. The test accuracy also stayed flat instead of falling, and with only 154 test patients it can move by 3 to 4 points from noise alone.

If overfitting did become a problem, there are a few ways to reduce it. Lowering `learning_rate` makes each stump have less influence, which slows the weight updates for hard samples. Using fewer estimators, for example stopping around 40 to 60 stumps where the curves flatten, gives about the same accuracy with a simpler model. Keeping the base learners weak, instead of using deeper trees, also limits overfitting. These settings are best chosen with cross-validation, for example `GridSearchCV`, not by looking at the test set, since that would leak information from it.